# Model Building

# Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report,accuracy_score,recall_score,r2_score,precision_score,roc_auc_score,confusion_matrix,\
f1_score,cohen_kappa_score

import warnings 
warnings.filterwarnings("ignore")

## Splitting the dataset

In [2]:
PROJECT_ROOT = Path.cwd().parent

train_df = pd.read_csv(PROJECT_ROOT / "data" / "train_sample_2024.csv")
test_df  = pd.read_csv(PROJECT_ROOT / "data" / "test_sample_2025.csv")

In [3]:
print(train_df.shape)
print(test_df.shape)

(200001, 25)
(100001, 25)


In [4]:
feature_cols = [
    'QUARTER',
    'MONTH',
    'DAY_OF_WEEK',
    'CRS_DEP_TIME',
    'DEP_TIME_BLK',
    'MKT_UNIQUE_CARRIER',
    'ORIGIN_AIRPORT_ID',
    'DEST_AIRPORT_ID',
    'DISTANCE',
    'DISTANCE_GROUP',
    'TAXI_OUT',
    'TAXI_IN',
    'AIR_TIME',
    'ACTUAL_ELAPSED_TIME'
]

train_num=train_df[feature_cols].select_dtypes(include=np.number).columns.to_list()
train_cat=train_df[feature_cols].select_dtypes(include=object).columns.to_list()

test_num=test_df[feature_cols].select_dtypes(include=np.number).columns.to_list()
test_cat=test_df[feature_cols].select_dtypes(include=object).columns.to_list()

## Data Preprocessing


In [5]:
le = LabelEncoder()
for col in train_cat:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    test_df[col] = le.transform(test_df[col])

In [6]:
train_features = ['QUARTER',
    'MONTH',
    'DAY_OF_WEEK',
    'CRS_DEP_TIME',
    'DEP_TIME_BLK',
    'MKT_UNIQUE_CARRIER',
    'ORIGIN_AIRPORT_ID',
    'DEST_AIRPORT_ID',
    'DISTANCE',
    'DISTANCE_GROUP',
    'TAXI_OUT',
    'TAXI_IN',
    'AIR_TIME',
    'ACTUAL_ELAPSED_TIME']

x_train = train_df[train_features]
y_train = train_df['OPERATIONAL_RISK']

In [7]:
test_features =['QUARTER',
    'MONTH',
    'DAY_OF_WEEK',
    'CRS_DEP_TIME',
    'DEP_TIME_BLK',
    'MKT_UNIQUE_CARRIER',
    'ORIGIN_AIRPORT_ID',
    'DEST_AIRPORT_ID',
    'DISTANCE',
    'DISTANCE_GROUP',
    'TAXI_OUT',
    'TAXI_IN',
    'AIR_TIME',
    'ACTUAL_ELAPSED_TIME']


x_test = test_df[test_features]
y_test=test_df['OPERATIONAL_RISK']

In [8]:
summary = pd.DataFrame(columns=['Name','Accuracy','Precision','Recall','F1_Score','ROC_AUC','Cohen_Kappa'])

def metrics(name, y_test, ypred_hard, ypred_soft):
    global summary

    print(f"\nClassification Report for {name}:\n")
    print(classification_report(y_test, ypred_hard))
    print('------------------------------------------------------------')

    acc = round(accuracy_score(y_test, ypred_hard), 2)
    pre = round(precision_score(y_test, ypred_hard), 2)
    rec = round(recall_score(y_test, ypred_hard), 2)
    f1  = round(f1_score(y_test, ypred_hard), 2)
    auc = round(roc_auc_score(y_test, ypred_soft), 2)
    reliability = round(cohen_kappa_score(y_test, ypred_hard), 2)

    result = pd.DataFrame({
        'Name':[name],
        'Accuracy':[acc],
        'Precision':[pre],
        'Recall':[rec],
        'F1_Score':[f1],
        'ROC_AUC':[auc],
        'Cohen_Kappa':[reliability]
    })

    summary = pd.concat([summary, result], ignore_index=True)
    return summary

## Baseline Model - Logistic Regression

In [9]:
model1 = LogisticRegression(random_state=42)
model1.fit(x_train, y_train)

ypred_soft_1 = model1.predict_proba(x_test)[:,1]
ypred_hard_1 = (ypred_soft_1 > 0.5).astype(int)

summary = metrics(
    "Baseline Logistic",
    y_test,
    ypred_hard_1,
    ypred_soft_1
)


Classification Report for Baseline Logistic:

              precision    recall  f1-score   support

           0       0.81      0.98      0.89     78229
           1       0.72      0.18      0.29     21772

    accuracy                           0.81    100001
   macro avg       0.77      0.58      0.59    100001
weighted avg       0.79      0.81      0.76    100001

------------------------------------------------------------


In [10]:
summary

,Name,Accuracy,Precision,Recall,F1_Score,ROC_AUC,Cohen_Kappa
0,Baseline Logistic,0.81,0.72,0.18,0.29,0.7,0.23


In [11]:
train_df.columns

Index(['YEAR', 'QUARTER', 'MONTH', 'DAY_OF_WEEK', 'MKT_UNIQUE_CARRIER',
       'ORIGIN_AIRPORT_ID', 'ORIGIN', 'DEST_AIRPORT_ID', 'DEST',
       'CRS_DEP_TIME', 'DEP_TIME_BLK', 'TAXI_OUT', 'TAXI_IN', 'CANCELLED',
       'DIVERTED', 'ACTUAL_ELAPSED_TIME', 'AIR_TIME', 'DISTANCE',
       'DISTANCE_GROUP', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY',
       'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY', 'OPERATIONAL_RISK'],
      dtype='str')

In [12]:
train_features

['QUARTER',
 'MONTH',
 'DAY_OF_WEEK',
 'CRS_DEP_TIME',
 'DEP_TIME_BLK',
 'MKT_UNIQUE_CARRIER',
 'ORIGIN_AIRPORT_ID',
 'DEST_AIRPORT_ID',
 'DISTANCE',
 'DISTANCE_GROUP',
 'TAXI_OUT',
 'TAXI_IN',
 'AIR_TIME',
 'ACTUAL_ELAPSED_TIME']

## Model 2 - Logistic Regression with class weights

In [13]:
# model2 = LogisticRegression(class_weight='balanced', random_state=42)
# model2.fit(x_train, y_train)

# ypred_soft_2 = model2.predict_proba(x_test)[:,1]
# ypred_hard_2 = (ypred_soft_2 > 0.5).astype(int)

# summary = metrics(
#     "Logistic class",
#     y_test,
#     ypred_hard_2,
#     ypred_soft_2
# )

In [14]:
# summary

In [15]:
# model3 = DecisionTreeClassifier(class_weight='balanced',random_state=42,max_depth=100)
# model3.fit(x_train, y_train)

# ypred_soft_3 = model3.predict_proba(x_test)[:,1]
# ypred_hard_3 = (ypred_soft_3 > 0.5).astype(int)

# summary = metrics(
#     "Decision Tree",
#     y_test,
#     ypred_hard_3,
#     ypred_soft_3
# )

In [16]:
summary

,Name,Accuracy,Precision,Recall,F1_Score,ROC_AUC,Cohen_Kappa
0,Baseline Logistic,0.81,0.72,0.18,0.29,0.7,0.23


In [17]:
# model4 = RandomForestClassifier(class_weight='balanced',random_state=42,max_depth=100)
# model4.fit(x_train, y_train)

# ypred_soft_4 = model4.predict_proba(x_test)[:,1]
# ypred_hard_4 = (ypred_soft_4 > 0.5).astype(int)

# summary = metrics(
#     "Random Forest",
#     y_test,
#     ypred_hard_4,
#     ypred_soft_4
# )

In [18]:
# summary

In [19]:
# model5 = XGBClassifier(
#     n_estimators=10000)

# model5.fit(x_train, y_train)
# ypred_soft_5 = model5.predict_proba(x_test)[:,1]
# ypred_hard_5 = (ypred_soft_5 > 0.5).astype(int)
# summary = metrics(
#     "XGBoost",
#     y_test,
#     ypred_hard_5,
#     ypred_soft_5
# )


In [20]:
summary

,Name,Accuracy,Precision,Recall,F1_Score,ROC_AUC,Cohen_Kappa
0,Baseline Logistic,0.81,0.72,0.18,0.29,0.7,0.23
